# Note 2: Partial Pivoting and LU Factorization

**Goal:** Fix the instability of naive GE by introducing row swaps, and recast elimination as a matrix factorization $PA = LU$.

## Recap: Why Naive GE Breaks

In Note 1 we saw two problems:
1. Zero pivots cause division by zero
2. Small pivots amplify rounding errors

Both are fixed by the same idea: before eliminating column $k$, **swap row $k$ with the row that has the largest entry in column $k$** (below the diagonal).

## Partial Pivoting

At step $k$, instead of using $a_{kk}$ as the pivot:
1. Find $p = \arg\max_{i \geq k} |a_{ik}|$
2. Swap rows $k$ and $p$
3. Proceed with elimination as before

This ensures every multiplier satisfies $|m_{ik}| \leq 1$, bounding error growth.

In [1]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)

In [2]:
def gauss_partial_pivot(A, b):
    """Solve Ax = b with Gaussian elimination + partial pivoting."""
    n = len(b)
    A = A.astype(float).copy()
    b = b.astype(float).copy()
    perm = list(range(n))  # track row permutation
    
    for k in range(n - 1):
        # Find the pivot: row with largest |A[i,k]| for i >= k
        pivot_row = k + np.argmax(np.abs(A[k:, k]))
        
        if abs(A[pivot_row, k]) < 1e-15:
            raise ValueError(f"Matrix is singular at column {k}")
        
        # Swap rows
        if pivot_row != k:
            A[[k, pivot_row]] = A[[pivot_row, k]]
            b[[k, pivot_row]] = b[[pivot_row, k]]
            perm[k], perm[pivot_row] = perm[pivot_row], perm[k]
        
        # Eliminate below pivot
        for i in range(k + 1, n):
            m = A[i, k] / A[k, k]
            A[i, k+1:] -= m * A[k, k+1:]
            b[i] -= m * b[k]
            A[i, k] = m  # store multiplier in the zeroed position
    
    # Back substitution
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (b[i] - A[i, i+1:] @ x[i+1:]) / A[i, i]
    
    return x, perm

## The System That Broke Naive GE

Let's revisit the failing examples from Note 1.

In [3]:
# Example 1: Zero pivot — now works!
A1 = np.array([[0.0, 1.0],
               [1.0, 1.0]])
b1 = np.array([1.0, 2.0])

x, perm = gauss_partial_pivot(A1, b1)
print(f"Zero-pivot system: x = {x}")
print(f"Residual: {np.linalg.norm(A1 @ x - b1):.2e}")

Zero-pivot system: x = [1. 1.]
Residual: 0.00e+00


In [4]:
# Example 2: Small pivot — now stable!
eps = 1e-15
A2 = np.array([[eps, 1.0],
               [1.0, 1.0]])
b2 = np.array([1.0, 2.0])

x_pp, _ = gauss_partial_pivot(A2, b2)
x_np = np.linalg.solve(A2, b2)

print(f"Partial pivoting: x = {x_pp}")
print(f"NumPy:            x = {x_np}")
print(f"PP residual:  {np.linalg.norm(A2 @ x_pp - b2):.2e}")
print(f"NumPy residual: {np.linalg.norm(A2 @ x_np - b2):.2e}")

Partial pivoting: x = [1. 1.]
NumPy:            x = [1. 1.]
PP residual:  0.00e+00
NumPy residual: 2.22e-16


## From Elimination to Factorization: $PA = LU$

Gaussian elimination with partial pivoting is equivalent to factoring:

$$PA = LU$$

where:
- $P$ is a permutation matrix (encodes row swaps)
- $L$ is unit lower triangular ($l_{ii} = 1$, $|l_{ij}| \leq 1$ for $i > j$)
- $U$ is upper triangular

**Why factorize?** Once we have $L$ and $U$, solving for a new right-hand side $b$ costs only $O(n^2)$ instead of $O(n^3)$:

$$Ax = b \implies PAx = Pb \implies LUx = Pb$$

1. Solve $Ly = Pb$ (forward substitution, $O(n^2)$)
2. Solve $Ux = y$ (back substitution, $O(n^2)$)

This is exactly what LAPACK's `dgesv` does — and what NumPy calls under the hood.

In [5]:
def lu_factor(A):
    """Compute PA = LU factorization with partial pivoting.
    
    Returns L, U, perm where perm[k] = row index of the k-th pivot.
    """
    n = A.shape[0]
    L = np.eye(n)
    U = A.astype(float).copy()
    perm = list(range(n))
    
    for k in range(n - 1):
        # Pivot selection
        pivot_row = k + np.argmax(np.abs(U[k:, k]))
        if pivot_row != k:
            U[[k, pivot_row]] = U[[pivot_row, k]]
            perm[k], perm[pivot_row] = perm[pivot_row], perm[k]
            # Also swap already-computed L entries
            L[[k, pivot_row], :k] = L[[pivot_row, k], :k]
        
        # Elimination
        for i in range(k + 1, n):
            L[i, k] = U[i, k] / U[k, k]
            U[i, k:] -= L[i, k] * U[k, k:]
    
    return L, U, perm


def lu_solve(L, U, perm, b):
    """Solve Ax = b given PA = LU factorization."""
    n = len(b)
    # Apply permutation: Pb
    pb = b[perm].astype(float)
    
    # Forward substitution: Ly = Pb
    y = np.zeros(n)
    for i in range(n):
        y[i] = pb[i] - L[i, :i] @ y[:i]
    
    # Back substitution: Ux = y
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]
    
    return x

In [6]:
# Demonstrate: factor once, solve twice
A = np.array([
    [2.0, 1.0, -1.0],
    [-3.0, -1.0, 2.0],
    [-2.0, 1.0, 2.0]
])

L, U, perm = lu_factor(A)

print("L =")
print(L)
print("\nU =")
print(U)
print(f"\nPermutation: {perm}")

# Verify PA = LU
P = np.eye(3)[perm]
print(f"\n||PA - LU|| = {np.linalg.norm(P @ A - L @ U):.2e}")

L =
[[ 1.        0.        0.      ]
 [ 0.666667  1.        0.      ]
 [-0.666667  0.2       1.      ]]

U =
[[-3.       -1.        2.      ]
 [ 0.        1.666667  0.666667]
 [ 0.        0.        0.2     ]]

Permutation: [1, 2, 0]

||PA - LU|| = 1.57e-16


In [7]:
# Solve with two different right-hand sides
b1 = np.array([8.0, -11.0, -3.0])
b2 = np.array([1.0, 0.0, 0.0])  # first column of A^{-1}

x1 = lu_solve(L, U, perm, b1)
x2 = lu_solve(L, U, perm, b2)

print(f"Solution 1: {x1}  (residual: {np.linalg.norm(A @ x1 - b1):.2e})")
print(f"Solution 2: {x2}  (residual: {np.linalg.norm(A @ x2 - b2):.2e})")

Solution 1: [ 2.  3. -1.]  (residual: 1.83e-15)
Solution 2: [ 4. -2.  5.]  (residual: 1.78e-15)


## Multiplier Growth and Stability

Partial pivoting ensures $|l_{ij}| \leq 1$ by construction. Let's verify this and show why it matters.

In [8]:
# Random 50x50 system — check multiplier bounds
n = 50
A_rand = np.random.randn(n, n)
L, U, perm = lu_factor(A_rand)

# All sub-diagonal entries of L should be <= 1
L_lower = np.tril(L, -1)
print(f"Max |L| entry (sub-diagonal): {np.max(np.abs(L_lower)):.6f}")
print(f"All |L_ij| <= 1? {np.all(np.abs(L_lower) <= 1.0 + 1e-10)}")

# Solve and check accuracy
b_rand = np.random.randn(n)
x = lu_solve(L, U, perm, b_rand)
print(f"\nRelative residual: {np.linalg.norm(A_rand @ x - b_rand) / np.linalg.norm(b_rand):.2e}")

Max |L| entry (sub-diagonal): 0.994005
All |L_ij| <= 1? True

Relative residual: 2.51e-15


## Timing: Factor Once, Solve Many

The factorization is $O(n^3)$, but each subsequent solve is only $O(n^2)$. This amortization is critical in optimization, where we solve the same-structure system at every iteration.

In [9]:
import time

n = 200
A = np.random.randn(n, n) + n * np.eye(n)

# Time the factorization
t0 = time.perf_counter()
L, U, perm = lu_factor(A)
t_factor = time.perf_counter() - t0

# Time 10 solves
t0 = time.perf_counter()
for _ in range(10):
    b = np.random.randn(n)
    x = lu_solve(L, U, perm, b)
t_solve = (time.perf_counter() - t0) / 10

print(f"n = {n}")
print(f"Factor: {t_factor:.4f}s")
print(f"Solve:  {t_solve:.4f}s  ({t_factor/t_solve:.0f}x cheaper)")

n = 200
Factor: 0.0260s
Solve:  0.0004s  (68x cheaper)


## What We've Learned

1. **Partial pivoting** (choosing the largest pivot in each column) eliminates both zero-pivot failures and small-pivot instability
2. GE with partial pivoting is equivalent to the **LU factorization** $PA = LU$
3. Factorization separates the $O(n^3)$ work from the $O(n^2)$ solve, enabling efficient re-use
4. All multipliers satisfy $|l_{ij}| \leq 1$, bounding error growth

## What's Next

LU works for general (non-symmetric) matrices. But in optimization, our matrices are **symmetric**: $A = A^T$. In Note 3, we exploit this symmetry with the **LDL^T factorization**, which halves the work and storage. We'll also handle the twist that our matrices are **indefinite** (not positive definite), requiring **Bunch-Kaufman pivoting** with $2 \times 2$ pivot blocks.

---

*This is Note 2 in a series building from basic Gaussian elimination to the multifrontal sparse solver used in [ripopt](https://github.com/jkitchin/ripopt).*